# 01. Adatok és leíró statisztika

## MIT mérünk?

Ez a fejezet a hirdetési adatbázis **összetételét** és a lakáspiac **alapvető szerkezetét** írja le: hány hirdetés van a mintában, ebből mennyi eladó és mennyi kiadó; hány hirdetés rendelkezik garantáltan pontos, épületszintű koordinátával; hogyan oszlanak meg a fajlagos árak (négyzetméterárak), a városrészek, a műszaki állapot, az építési korszak és a szobaszám szerint; milyen együttmozgások (korrelációk) vannak a kulcsváltozók között; végül pedig milyen „természetes" lakástípusok (klaszterek) rajzolódnak ki az adatokból.

## MIÉRT fontos?

Minden későbbi térbeli és ökonometriai elemzés csak akkor értelmezhető, ha pontosan tudjuk, miből áll a minta. Különösen kritikus a **koordináta-őszinteség**: a hirdetési platformok néha csak megosztott vagy utcaszintű (nem épületszintű) koordinátát mellékelnek, és az ilyen hirdetésekre számolt méter-pontos távolságok hamis pontosságot sugallnának. Ezért a térbeli mezők (koordináták, hálózati távolságok, ellátottsági indexek) kizárólag az A-osztályú, garantáltan pontos almintán vannak kitöltve; a többi rekordnál szigorúan üresek. A leíró statisztika emellett megmutatja az áreloszlás alakját is — ez dönti el, hogy az ár jellemzésére átlagot vagy mediánt érdemes használni.

## HOGYAN mérünk?

A fejezet minden számítása egy közös elemző modulból, gyorsítótárból érkezik: a notebook nem illeszt modelleket és nem számol újra elemzést, csak megjeleníti a kész eredményeket. A mintát három szinten vizsgáljuk: a **teljes adatbázis** (df), az **eladó hirdetések** és a **garantáltan pontos, eladó elemzési minta** (p) — ez utóbbi a kutatás tényleges elemzési egysége. Az árakat fajlagos formában (négyzetméterár) hasonlítjuk össze, mert így a különböző méretű lakások összemérhetők.

## Mi a döntési szabály?

Minden állítást úgy fogalmazunk meg, hogy a hivatkozott szám a gyorsítótárból, futásidőben épül be a szövegbe — nincs előre beírt érték. Az ár jellemzésére a **mediánt** használjuk, mert a kínálati árak jellemzően jobbra ferdék (kevés, nagyon drága hirdetés húzza felfelé az átlagot). A városrészeket és a kategóriákat sosem nevezzük meg előre: minden bontás a `varosresz`, `allapot` és a többi kategóriaoszlop tényleges értékeiből képződik, így a notebook bármely vizsgált területre változtatás nélkül futtatható.


In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys, os; sys.path.insert(0, '..')
from ingatlan_tdk._utils import *
from ingatlan_tdk import analyses as A
setup_plotly()
AREA = os.environ.get('TDK_ACTIVE_AREA', 'kobanya')
df = load_szamitott_master(); p = A.pontos_elado()


In [ ]:
import plotly.express as px
# A terület neve és fejléce dinamikusan a konfigurációból
from IPython.display import display, HTML, Markdown

area_name = get_area_metadata(os.environ.get('TDK_ACTIVE_AREA', 'kobanya'))['name']
display(HTML(render_area_header_html(area=AREA, df=df)))
print(f"Aktív terület: {area_name}  |  teljes minta: {len(df):,} sor, {len(df.columns)} oszlop  |  elemzési minta (pontos, eladó): {len(p):,} sor".replace(',', ' '))


## 1. Adatforrások és a koordináta-őszinteség elve

**MIT mérünk?** Azt, hogy a hirdetési adatbázis milyen geokódolási minőségű rekordokból áll, és ebből mekkora az a részhalmaz, amelyre térbeli számításokat szabad végezni. A hirdetési adatokat térinformatikai rétegekkel (utcahálózat, kötöttpályás hálózat, szolgáltatási pontok) egészítjük ki, és a nyers adatok szigorú tisztításon mennek át (duplikációszűrés, szélsőérték-kezelés, geokódolás).

**MIÉRT?** Egy ingatlan „pontos" helyét csak akkor ismerjük, ha a koordináta az épülethez — nem utcaközéphez, nem városrész-központhoz — köthető. A hirdetési platformok gyakran adnak meg koordinátát úgy, hogy több hirdetés ugyanazt a pontot kapja (megosztott koordináta), vagy utcaszintű geokódolást alkalmaznak — ilyenkor egy „méter-pontos" távolság vagy sétaizokrón hamis precizitás lenne.

**HOGYAN?** A feldolgozás minden rekordhoz `geokodolas_pontossag` besorolást rendel: az **A**-osztály (épületszintű, egyedi vagy hivatalos geokódolás) garantáltan pontos, a **B**-osztály (megosztott/aggregált koordináta) és a koordináta nélküli rekordok nem azok. A `minta_garantalt_pontos` zászló éppen az A-osztályú rekordokat jelöli: csak ezekre töltjük ki a koordinátákat, a hálózati távolságokat, az izokrón- és POI-mezőket; minden más rekordnál ezek a mezők üresek maradnak.

**Döntési szabály:** a térbeli állítások kizárólag a garantáltan pontos almintán érvényesek; a nem pontos rekordok a leíró (nem térbeli) statisztikákban szerepelhetnek, de távolságalapú elemzésbe nem kerülnek.


In [ ]:
# 1. Minta-KPI-k és a geokódolási osztályok — minden szám a cache-ből (A.descriptive)
D = A.descriptive()
m = D['minta'].set_index('mutato')['ertek']

n_osszes = int(m['osszes']); n_elado = int(m['elado']); n_kiado = int(m['kiado'])
n_pontos = int(m['pontos']); n_pontos_elado = int(m['pontos_elado'])
lt = D['listing_type_ar'].set_index('listing_type')
med_elado = float(lt.loc['elado', 'median']) if 'elado' in lt.index else float('nan')
med_kiado = float(lt.loc['kiado', 'median']) if 'kiado' in lt.index else float('nan')

kpi_cards = [
    ("Összes hirdetés", f"{n_osszes:,} db".replace(',', ' '), "Teljes adatbázis", "#1e3a8a"),
    ("Eladó hirdetés", f"{n_elado:,} db".replace(',', ' '), f"{n_elado/n_osszes*100:.1f}% részarány", "#2563eb"),
    ("Kiadó hirdetés", f"{n_kiado:,} db".replace(',', ' '), f"{n_kiado/n_osszes*100:.1f}% részarány", "#d97706"),
    ("Garantáltan pontos", f"{n_pontos:,} db".replace(',', ' '), f"{n_pontos/n_osszes*100:.1f}% A-osztályú koordináta", "#16a34a"),
    ("Elemzési minta (pontos eladó)", f"{n_pontos_elado:,} db".replace(',', ' '), f"{n_pontos_elado/n_elado*100:.1f}% az eladókból", "#9333ea"),
    ("Eladó medián ár/m²", fmt_huf(med_elado), "Fajlagos kínálati ár", "#0891b2"),
]
display(HTML(kpi_grid_html(kpi_cards)))

# A geokódolási osztályok tényleges összetétele (dinamikusan az oszlop értékeiből)
acc = (df['geokodolas_pontossag'].fillna('ismeretlen').value_counts()
       .rename_axis('geokodolas_pontossag').reset_index(name='N'))
acc['reszarany_pct'] = (acc['N'] / acc['N'].sum() * 100).round(1)
display(HTML("<b>Geokódolási osztályok a teljes mintán (geokodolas_pontossag):</b><br>" +
             acc.to_html(classes='table table-bordered table-striped', index=False)))
a_osszes = int(acc.loc[acc['geokodolas_pontossag'].str.startswith('A'), 'N'].sum())
print(f"A teljes minta {n_osszes} sorából {a_osszes} rendelkezik A-osztályú (épületszintű) koordinátával "
      f"({a_osszes/n_osszes*100:.1f}%); ezeket jelöli a minta_garantalt_pontos zászló, "
      f"a többi {n_osszes-a_osszes} rekord térbeli mezői üresek maradnak.")


## 2. A kanonikus változók áttekintése (adatkonyv)

**MIT mérünk?** A kutatás kanonikus (egységes) változókészletét mutatjuk be: mi a technikai oszlopnév, mit jelent, milyen mértékegységű, és mi a szerepe a kutatásban.

**MIÉRT?** A feldolgozott adathalmaz sok tucat oszlopot tartalmaz; a későbbi fejezetek csak erre a kanonikus készletre épülnek (például a hedonikus modellek fizikai kontrolljai vagy a térbeli mérések sávrendszerei). Egy rövid adatkonyv megakadályozza, hogy ugyanazt a fogalmat különböző oszlopokkal keverjük össze.

**HOGYAN?** Az alábbi táblázat a legfontosabb oszlopokat csoportosítja (pénzügyi célváltozók, fizikai jellemzők, hálózati elérhetőség, környezeti externáliák, POI-ellátottság, térbeli azonosítók), és mindegyikhez megadja a kitöltöttséget. Fontos: a hálózati távolság-, izokrón-, immisszió- és POI-oszlopok **strukturálisan** csak a garantáltan pontos almintán vannak kitöltve — ez nem adathiány, hanem a koordináta-őszinteségi szabály következménye.

**Döntési szabály:** egy elemzés akkor kerül a kanonikus készletbe, ha a vizsgált területek adatbázisaiban ugyanazzal a névvel és definícióval létezik; a táblázat csak a ténylegesen létező oszlopokat sorolja fel.


In [ ]:
# 2. Kanonikus változók adatkonyva (a legfontosabb oszlopok; kitöltöttség + szerep)
valtozo_meta = [
    # Pénzügyi célváltozók
    ('price_huf', 'Kínálati vételár', 'Pénzügyi', 'HUF', 'Célváltozó (bruttó összeg)'),
    ('nm_ar_huf', 'Négyzetméterár', 'Pénzügyi', 'HUF/m²', 'Fajlagos célváltozó'),
    ('log_nm_ar', 'Fajlagos ár logaritmusa', 'Pénzügyi', 'ln(HUF/m²)', 'Ökonometriai regressziós cél'),
    ('ar_millio_ft', 'Vételár millió Ft-ban', 'Pénzügyi', 'M Ft', 'Leíró statisztikai mutató'),
    # Fizikai és épületjellemzők
    ('alapterulet_nm', 'Alapterület', 'Fizikai', 'm²', 'Fizikai alaptulajdonság'),
    ('korrigalt_alapterulet_nm', 'Korrigált alapterület', 'Fizikai', 'm²', 'Fél erkéllyel súlyozva'),
    ('szobaszam_osszes', 'Összes szobaszám', 'Fizikai', 'db', 'Belső beosztási kontroll'),
    ('allapot_kod', 'Műszaki állapot index', 'Fizikai', '1-6 skála', 'Hedonikus minőségi rang'),
    ('is_panel', 'Panelszerkezet dummy', 'Fizikai', '0/1', 'Technológiai diszkont'),
    ('has_lift', 'Lift megléte dummy', 'Fizikai', '0/1', 'Kényelmi felszereltség'),
    ('has_erkely', 'Erkély megléte dummy', 'Fizikai', '0/1', 'Kültéri kapcsolat prémiuma'),
    ('emelet_szam', 'Emelet szintszám', 'Fizikai', 'szint', 'Függőleges elhelyezkedés'),
    ('epulet_kora_ev', 'Épület becsült kora', 'Fizikai', 'év', 'Amortizációs hatás'),
    ('epites_eve_kategoria', 'Építési korszak', 'Fizikai', 'korszak', 'Építészeti korcsoport'),
    # Hálózati közlekedési távolságok
    ('tavolsag_metro_halozati_m', 'Metróállomás hálózati táv.', 'Hálózati elérhetőség', 'méter', 'Gyalogos metróelérés'),
    ('tavolsag_vasut_halozati_m', 'Vasútállomás hálózati táv.', 'Hálózati elérhetőség', 'méter', 'Állomási elérhetőség (TOD)'),
    ('tavolsag_villamos_halozati_m', 'Villamosmegálló hálózati táv.', 'Hálózati elérhetőség', 'méter', 'Felszíni kötöttpályás hálózat'),
    ('tavolsag_kotottpalya_halozati_m', 'Kötöttpálya-index (hálózat)', 'Hálózati elérhetőség', 'méter', 'min(metró, vasút, villamos) hálózati táv'),
    # Környezeti externáliák
    ('tavolsag_vasut_m', 'Vasúti pálya légvonal', 'Környezeti externália', 'méter', 'Immissziós zaj- és rezgésterhelés'),
    ('vasut_zona', 'Vasúti immissziós sáv', 'Környezeti externália', 'kategória', 'A rögzített 0-150-300-500-1000-2000 m sáv'),
    ('is_vasut_immisszio_150m', 'Közvetlen immisszió (<150 m)', 'Környezeti externália', '0/1', 'Immissziós zóna dummy'),
    # POI-sávok (hálózati elérhetőség)
    ('poi_5p_count', 'POI 5p (≤375 m) hálózat', 'POI ellátottság', 'db', 'Közvetlen környezet szolgáltatásai'),
    ('poi_10p_count', 'POI 10p (≤750 m) hálózat', 'POI ellátottság', 'db', 'Napi szükségletek elérhetősége'),
    ('poi_15p_count', 'POI 15p (≤1125 m) hálózat', 'POI ellátottság', 'db', '15 perces város ellátottság'),
    # Térbeli GIS azonosítók
    ('geokodolt_lat', 'WGS84 Szélesség (Lat)', 'Térbeli GIS', 'fok', 'Térbeli pontkoordináta'),
    ('geokodolt_lon', 'WGS84 Hosszúság (Lon)', 'Térbeli GIS', 'fok', 'Térbeli pontkoordináta'),
    ('varosresz', 'Városrész', 'Térbeli GIS', 'név', 'Városrészi szegmens'),
    ('minta_garantalt_pontos', 'Pontos koordináta zászló', 'Térbeli GIS', '0/1', 'A-osztályú (épületszintű) minta jelzője'),
]

df_pontos = df[df['minta_garantalt_pontos'] == 1]
stats_rows, dict_rows = [], []
for col, hu_name, kat, unit, role in valtozo_meta:
    if col not in df.columns:
        continue
    structural = (col.startswith('tavolsag_') or col.startswith('poi_')
                  or col in ('vasut_zona', 'is_vasut_immisszio_150m'))
    base = df_pontos if structural else df
    n_valid = int(base[col].notna().sum())
    pct = (n_valid / len(base)) * 100
    note = ' (pontos almintán)' if structural else ''
    stats_rows.append({'Technikai oszlop': col, 'Változó megnevezése': hu_name + note,
                       'Kategória': kat, 'Kitöltöttség (%)': round(pct, 1), 'Érvényes N': n_valid})
    dict_rows.append({'Kategória': kat, 'Változó neve': hu_name,
                      'Technikai név': f"<code>{col}</code>", 'Mértékegység': unit,
                      'Érvényes (db)': f"{n_valid:,} ({pct:.1f}%)".replace(',', ' '),
                      'Szerepe a kutatásban': role})

df_qual = pd.DataFrame(stats_rows).sort_values(['Kategória', 'Kitöltöttség (%)'], ascending=[True, True])

fig2 = px.bar(
    df_qual, x='Kitöltöttség (%)', y='Változó megnevezése', color='Kategória',
    orientation='h',
    title=f'Kutatási Változók Kitöltöttsége Kategóriánként (N = {len(df):,} db)'.replace(',', ' '),
    labels={'Kitöltöttség (%)': 'Kitöltöttség aránya (%)', 'Változó megnevezése': 'Változó'},
    range_x=[0, 105], template=PLOTLY_TEMPLATE, height=800
)
fig2.update_layout(yaxis=dict(tickfont=dict(size=10)), margin=dict(l=220, r=30, t=50, b=50))
fig2.show()

df_dict = pd.DataFrame(dict_rows)
html_dict = ("<div style='overflow-x:auto; margin: 20px 0; max-height: 480px; overflow-y: auto;'>"
             + df_dict.to_html(classes='table table-bordered table-striped table-hover', index=False, escape=False) + "</div>")
display(HTML("<b>Kanonikus adatkonyv — a legfontosabb változók:</b>" + html_dict))
display(HTML("<p style='color:#64748b; font-size:12px;'>Megjegyzés: a hálózati távolság-, izokrón-, immisszió- és POI-változók kitöltöttsége a garantáltan pontos almintán értelmezett (strukturális hiány, nem adathiány).</p>"))


## 3. Árstruktúra: hirdetéstípusok, városrészek, decilisek

**MIT mérünk?** A fajlagos négyzetméterárak megoszlását három bontásban: (1) hirdetéstípus (eladó/kiadó) szerinti árstatisztika a teljes mintán, (2) városrészek szerinti medián- és átlagárak az elemzési mintán, (3) az elemzési minta decilisei — a rendezett árlista tizedelő pontjai.

**MIÉRT?** A hirdetéstípus szerinti bontás azt mutatja, mennyire eltérő árszinten mozog az eladási és a bérleti piac; a városrész-mediánok a belső térszerkezet árbeli tagoltságát; a decilisek pedig az eloszlás farkait — hol kezdődik a legolcsóbb tized, és hol a legdrágább.

**HOGYAN?** Mindhárom táblázat a közös elemző modul `descriptive()` eredményéből készül (gyorsítótárból); a városrész-bontás a `varosresz` oszlop tényleges értékeiből képződik. A mediánt és az átlagot együtt közöljük: a kettő eltérése maga is diagnosztika (a medián alacsonyabb az átlagnál, ha az eloszlás jobbra ferde).

**Döntési szabály:** az ár-állítások alapja a medián; az átlagot csak a ferdeség érzékeltetésére használjuk. A városrész-sorrend a medián szerinti rendezés — ez adatvezérelt, nem előzetes besorolás.


In [ ]:
# 3. Árstatisztikák a cache-ből: hirdetéstípus, városrész, decilisek
def huf_fmt(v):
    return f"{v:,.0f} Ft/m²".replace(',', ' ') if pd.notna(v) else "—"

lt2 = D['listing_type_ar'].copy()
lt2.columns = ['Hirdetés típusa', 'N', 'Átlag (Ft/m²)', 'Medián (Ft/m²)', 'Min (Ft/m²)', 'Max (Ft/m²)']
display(HTML("<b>Hirdetéstípus szerinti árstatisztika (teljes minta, nm_ar_huf):</b><br>"
             + lt2.to_html(classes='table table-bordered table-striped', index=False,
                           float_format=lambda v: f"{v:,.0f}")))

vr = D['varosresz'].copy()
vr.columns = ['Városrész', 'N', 'Medián (Ft/m²)', 'Átlag (Ft/m²)']
display(HTML("<b>Városrész-mediánok (elemzési minta, medián szerint rendezve):</b><br>"
             + vr.to_html(classes='table table-bordered table-striped', index=False,
                          float_format=lambda v: f"{v:,.0f}")))

fig_vr = px.bar(vr, x='Medián (Ft/m²)', y='Városrész', orientation='h',
                color='Medián (Ft/m²)', color_continuous_scale='Blues', text='N',
                title='Városrészek medián négyzetméterára (elemzési minta)',
                labels={'Medián (Ft/m²)': 'Medián ár / m² (Ft)', 'N': 'Elemszám'},
                template=PLOTLY_TEMPLATE, height=420)
fig_vr.update_layout(coloraxis_showscale=False)
fig_vr.show()

dec = D['decilisek']
fig_dec = px.bar(dec, x='decilis', y='nm_ar_huf', color='nm_ar_huf', color_continuous_scale='RdBu_r',
                 text=dec['nm_ar_huf'].round(0).map(lambda v: f"{v:,.0f}"),
                 title='Az elemzési minta árdecilisei (Ft/m²)',
                 labels={'decilis': 'Decilis', 'nm_ar_huf': 'Ár / m² (Ft)'},
                 template=PLOTLY_TEMPLATE, height=400)
fig_dec.update_layout(coloraxis_showscale=False)
fig_dec.show()

top_vr = vr.iloc[0]; bot_vr = vr.iloc[-1]
print(f"A legmagasabb medián árú városrész: {top_vr['Városrész']} ({huf_fmt(top_vr['Medián (Ft/m²)'])}, N={int(top_vr['N'])}); "
      f"a legalacsonyabb: {bot_vr['Városrész']} ({huf_fmt(bot_vr['Medián (Ft/m²)'])}, N={int(bot_vr['N'])}). "
      f"A mediánár-terjedelem a két végpont között {top_vr['Medián (Ft/m²)']/bot_vr['Medián (Ft/m²)']:.2f}-szeres.")


## 4. Eloszlások: állapot, korszak, szobaszám

**MIT mérünk?** Az elemzési minta összetételét három kategorikus dimenzió mentén: a lakások műszaki állapota, az épület kora (építési korszak) és a szobaszám-kategória.

**MIÉRT?** Ezek a változók a későbbi hedonikus modellek kulcskontrolljai. Ha például a minta döntően egy állapot- vagy korszak-kategóriára koncentrálódik, az a modellek azonosíthatóságát is befolyásolja — a leíró bontás már itt láthatóvá teszi, mely kategóriákból van elegendő megfigyelés.

**HOGYAN?** A kategória-gyakoriságok a közös elemző modul `descriptive()` eredményéből jönnek (gyorsítótár); a notebook oszlopdiagramokon jeleníti meg őket, a tényleges kategórianeveket a megfelelő oszlopok értékeiből véve.

**Döntési szabály:** nincs előzetes „helyes" megoszlás — a fejezet azt rögzíti, milyen a minta tényleges szerkezete; a kategóriák sorrendjét az adat (gyakoriság) adja.


In [ ]:
# 4. Kategorikus eloszlások a cache-ből
def cat_bar(tbl, xcol, title, ytitle, height=360):
    t = tbl.copy().sort_values('N', ascending=False)
    fig = px.bar(t, x=xcol, y='N', color='N', color_continuous_scale='Blues',
                 text='N', title=title, labels={xcol: ytitle, 'N': 'N'},
                 template=PLOTLY_TEMPLATE, height=height)
    fig.update_layout(coloraxis_showscale=False)
    return fig

fig_all = cat_bar(D['allapot'], 'allapot', 'Műszaki állapot szerinti megoszlás (elemzési minta)', 'Állapot')
fig_all.show()
fig_kor = cat_bar(D['korszak'], 'korszak', 'Építési korszak szerinti megoszlás (elemzési minta)', 'Korszak')
fig_kor.show()
fig_szo = cat_bar(D['szobaszam'], 'kategoria', 'Szobaszám-kategória szerinti megoszlás (elemzési minta)', 'Szobaszám-kategória')
fig_szo.show()

domin = D['allapot'].sort_values('N', ascending=False).iloc[0]
domin_kor = D['korszak'].sort_values('N', ascending=False).iloc[0]
print(f"A leggyakoribb állapotkategória: „{domin['allapot']}” ({int(domin['N'])} lakás, "
      f"{domin['N']/D['allapot']['N'].sum()*100:.1f}% az elemzési mintából). "
      f"A leggyakoribb korszak-kategória: „{domin_kor['korszak']}” ({int(domin_kor['N'])} lakás).")


## 5. Korrelációs hőtérkép

**MIT mérünk?** A kulcsváltozók páronkénti együttmozgását az elemzési mintán: Pearson-korreláció a lineáris kapcsolatra, Spearman-rangkorreláció a monoton (akár nemlineáris) kapcsolatra, szignifikanciaszintekkel.

**MIÉRT?** A hőtérkép megmutatja, mely változók mozognak együtt (például a lakásméret és az összár), és előre jelzi a későbbi modellek kollinearitási kockázatait. Azt is láthatóvá teszi, hogy az árral mely jellemzők korrelálnak pozitívan (állapot, ellátottság) és negatívan (épületkor).

**HOGYAN?** A mátrix az elemzési minta számszerű oszlopain számolódik; a cellák a korrelációs együtthatót és a p-értékből képzett csillagokat (* p<0,05, ** p<0,01, *** p<0,001) mutatják. A listwise minta csak a teljes kitöltöttségű sorokat tartalmazza.

**Döntési szabály:** a szignifikáns (p<0,05) együtthatók kapnak csillagot; a hőtérkép diagnosztikai eszköz, nem önálló hatásbecslés — az oksági állításokat a későbbi modellek végzik.


In [ ]:
# 5. Korrelációs hőtérkép az elemzési mintán (p), szignifikancia-csillagokkal
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

corr_vars = [
    'nm_ar_huf', 'alapterulet_nm', 'szobaszam_osszes', 'epulet_kora_ev',
    'allapot_kod', 'tavolsag_vasut_m', 'tavolsag_vasut_halozati_m', 'poi_15p_count'
]
corr_vars = [c for c in corr_vars if c in p.columns and p[c].dtype.kind in 'iuf']
corr_labels = {
    'nm_ar_huf': 'Ár / m²', 'alapterulet_nm': 'Alapterület', 'szobaszam_osszes': 'Szobaszám',
    'epulet_kora_ev': 'Épület kora', 'allapot_kod': 'Állapot kód',
    'tavolsag_vasut_m': 'Vágány légvonal', 'tavolsag_vasut_halozati_m': 'Állomás hálózati',
    'poi_15p_count': 'POI-k 15 perc',
}
corr_df = p[corr_vars].dropna()

def corr_pvalues(df, method):
    cols = list(df.columns)
    pmat = pd.DataFrame(np.eye(len(cols)), index=cols, columns=cols)
    fn = stats.pearsonr if method == 'pearson' else stats.spearmanr
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            res = fn(df.iloc[:, i], df.iloc[:, j])
            pv = res.pvalue if hasattr(res, 'pvalue') else res[1]
            pmat.iloc[i, j] = pv; pmat.iloc[j, i] = pv
    return pmat

def star_cells(rho, pmat):
    txt = pd.DataFrame('', index=rho.index, columns=rho.columns)
    for i in rho.index:
        for j in rho.columns:
            pv = pmat.loc[i, j]
            stars = '***' if pv < 0.001 else ('**' if pv < 0.01 else ('*' if pv < 0.05 else ''))
            txt.loc[i, j] = f"{rho.loc[i, j]:.2f}{stars}"
    return txt

pearson_r = corr_df.corr(method='pearson')
pearson_p = corr_pvalues(corr_df, 'pearson')
spearman_r = corr_df.corr(method='spearman')
spearman_p = corr_pvalues(corr_df, 'spearman')

fig_corr = make_subplots(
    rows=1, cols=2,
    subplot_titles=(f'Pearson-korreláció (n = {len(corr_df)})',
                    f'Spearman-rangkorreláció (n = {len(corr_df)})'),
    horizontal_spacing=0.25)
for i, (rho, pmat) in enumerate([(pearson_r, pearson_p), (spearman_r, spearman_p)], start=1):
    sub = px.imshow(rho, zmin=-1, zmax=1, color_continuous_scale='RdBu_r', text_auto=False)
    tr = sub.data[0]
    if getattr(tr, 'coloraxis', None) and getattr(sub.layout, 'coloraxis', None) is not None:
        tr.colorscale = sub.layout.coloraxis.colorscale
        tr.coloraxis = None
    tr.zmin = -1; tr.zmax = 1
    tr.text = star_cells(rho, pmat).values
    tr.texttemplate = '%{text}'
    tr.x = [corr_labels.get(c, c) for c in rho.columns]
    tr.y = [corr_labels.get(c, c) for c in rho.index]
    fig_corr.add_trace(tr, row=1, col=i)

fig_corr.update_layout(
    title_text=(f'Korrelációs hőtérkép (elemzési minta, listwise n = {len(corr_df)}) — '
                '* p < 0.05, ** p < 0.01, *** p < 0.001'),
    template=PLOTLY_TEMPLATE, height=580)
fig_corr.update_xaxes(tickangle=-30)
fig_corr.show()

if len(corr_df) and 'nm_ar_huf' in corr_df.columns:
    rho_nm_meret = pearson_r.loc['nm_ar_huf', 'alapterulet_nm']
    rho_nm_kor = pearson_r.loc['nm_ar_huf', 'epulet_kora_ev']
    print(f"A fajlagos ár és az alapterület Pearson-korrelációja r = {rho_nm_meret:.2f}, "
          f"a fajlagos ár és az épületkor korrelációja r = {rho_nm_kor:.2f} (n = {len(corr_df)}).")


## 6. Klaszterprofilok: a piac természetes csoportjai

**MIT mérünk?** A standardizált K-means eljárás (K=4 klaszter, rögzített véletlen mag) a kanonikus jellemzők (log fajlagos ár, korrigált alapterület, szobaszám, állapot, épületkor, emelet) alapján csoportokba rendezi a lakásokat; a fejezet a csoportok átlagos profilját és elemszámát mutatja.

**MIÉRT?** A klaszterek a piac „archetípusait" tárják fel: mely jellemző-kombinációk fordulnak elő ténylegesen (például kicsi és új építésű, magas állapotú lakások kontra nagy, idős, felújítandó lakások). A tipológia a későbbi szegmens-specifikus modellek alapja.

**HOGYAN?** A klaszterezés a közös elemző modulban futott le (gyorsítótárból olvassuk be); a notebook a profil-táblát (klaszter × változó átlag + elemszám) jeleníti meg, és oszlopdiagramon hasonlítja össze a standardizált profilokat, hogy a különböző mértékegységű változók egy skálán látszódjanak.

**Döntési szabály:** a klaszterszám a kutatási keretben rögzített K=4; a csoportok címkézése adatvezérelt (a profilértékekből), nem előzetes narratíva.


In [ ]:
# 6. Klaszterprofilok a cache-ből (A.cluster_analysis)
C = A.cluster_analysis()
prof = C['profil'].copy()
n_clus = int(C['n']['ertek'].iloc[0])

display(HTML(f"<b>Klaszterprofilok — K=4 K-means a kanonikus jellemzőkön (n = {n_clus}):</b><br>"
             + prof.round(2).to_html(classes='table table-bordered table-striped', index=False)))

# Standardizált profil-oszlopdiagram (a változók egy skálán összehasonlíthatók)
var_cols = [c for c in prof.columns if c not in ('klaszter', 'N')]
prof_std = prof[var_cols].copy()
prof_std = (prof_std - prof_std.mean()) / prof_std.std(ddof=0).replace(0, 1)
prof_std['klaszter'] = prof['klaszter'].map(lambda k: f"K{int(k)+1}")
prof_std['N'] = prof['N'].map(lambda n: f"N={int(n)}")
prof_long = prof_std.melt(id_vars=['klaszter', 'N'], var_name='változó', value_name='standardizált átlag')
fig_cl = px.bar(prof_long, x='változó', y='standardizált átlag', color='klaszter', barmode='group',
                hover_data={'N': True},
                title='Standardizált klaszterprofilok (0 = változónkénti mintaátlag)',
                template=PLOTLY_TEMPLATE, height=440)
fig_cl.update_layout(legend_title_text='Klaszter')
fig_cl.show()

n_cols = prof[['klaszter', 'N']].sort_values('klaszter')
sizes = ", ".join(f"K{int(r['klaszter'])+1} = {int(r['N'])}" for _, r in n_cols.iterrows())
print(f"A klaszterezés {n_clus} megfigyelésre illeszkedett; a klaszterméretek: {sizes}. "
      f"A legnagyobb klaszter a minta {n_cols['N'].max()/n_cols['N'].sum()*100:.1f}%-át adja.")


## 7. Interaktív adatszűrő (kliensoldali vezérlőpult)

**MIT mérünk?** A teljes minta szabadon szűrhető hirdetéstípus, térbeli pontosság és városrész szerint; a panel valós időben számolja a szűrt minta elemszámát, ár-mediánját és -átlagát, fajlagos árát és alapterületét, városrészi bontással.

**MIÉRT?** A leíró táblák rögzített bontásokat mutatnak; a szűrő lehetővé teszi tetszőleges kombinációk (például „csak eladó, csak pontos, egy adott városrész") azonnali ellenőrzését anélkül, hogy új elemzést kellene futtatni.

**HOGYAN?** A motor teljes egészében a böngészőben fut (JavaScript): a mikroadat-sorok a notebook futtatásakor beágyazódnak a cella kimenetébe, és minden szűrőváltás helyben újraszámolja a mutatókat. A medián és az átlag egyszerre látszik, így a ferdeség is érzékelhető.

**Döntési szabály:** a szűrő csak megjelenítés — nem módosítja az elemzési mintát, és a kutatási állítások továbbra is a gyorsítótárban rögzített számokon alapulnak.


In [ ]:
# 7. Valós idejű kliensoldali adathalmaz-szűrő (a teljes mintán: típus / pontosság / városrész)
import json

# Mikroadat a kliensoldali motorhoz: a szűrt minta VALÓDI mediánjának és átlagának számításához
microdata = []
for r in df.itertuples(index=False):
    if pd.isna(r.price_huf) or pd.isna(r.nm_ar_huf) or pd.isna(r.alapterulet_nm):
        continue
    microdata.append({
        't': str(r.listing_type), 'vr': str(r.varosresz), 'pt': int(r.minta_garantalt_pontos),
        'p': float(r.price_huf), 'nm': float(r.nm_ar_huf), 'a': float(r.alapterulet_nm)
    })

vr_list = sorted(list(df['varosresz'].dropna().unique()))

html_nb01 = f'''
<div id="nb01_filter_app" style="background:#ffffff; border:1px solid #cbd5e1; border-radius:12px; padding:22px; box-shadow:0 4px 6px -1px rgba(0,0,0,0.06); margin:18px 0; font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Helvetica,Arial,sans-serif;">
  <div style="display:flex; justify-content:space-between; align-items:center; border-bottom:2px solid #f1f5f9; padding-bottom:12px; margin-bottom:18px;">
    <div>
      <h3 style="margin:0; color:#1e3a8a; font-size:19px; font-weight:700;">🔎 Interaktív Adathalmaz Szűrő & Mintavételi Vezérlőpult</h3>
      <p style="margin:3px 0 0 0; color:#64748b; font-size:13px;">Dinamikus szegmentáció és statisztikai összegzés a teljes ingatlanpiaci mintán</p>
    </div>
    <span style="background:#dbeafe; color:#1d4ed8; font-size:11px; font-weight:700; padding:4px 10px; border-radius:9999px;">Kliensoldali JS Motor</span>
  </div>

  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(260px, 1fr)); gap:18px; margin-bottom:18px;">
    <!-- Típus és Pontosság szűrő -->
    <div style="background:#f8fafc; padding:14px; border-radius:8px; border:1px solid #e2e8f0;">
      <div style="font-weight:700; color:#334155; margin-bottom:10px; font-size:13px; text-transform:uppercase;">1. Hirdetés Típusa:</div>
      <div style="display:flex; gap:12px; margin-bottom:14px;">
        <label style="font-size:13px; font-weight:600; color:#334155; cursor:pointer;">
          <input type="radio" name="nb01_type" value="all" checked onchange="filterNB01()"> Mindkettő
        </label>
        <label style="font-size:13px; font-weight:600; color:#2563eb; cursor:pointer;">
          <input type="radio" name="nb01_type" value="elado" onchange="filterNB01()"> Eladó lakások
        </label>
        <label style="font-size:13px; font-weight:600; color:#d97706; cursor:pointer;">
          <input type="radio" name="nb01_type" value="kiado" onchange="filterNB01()"> Kiadó lakások
        </label>
      </div>

      <div style="font-weight:700; color:#334155; margin-bottom:8px; font-size:13px; text-transform:uppercase;">2. Térbeli Pontosság:</div>
      <label style="display:flex; align-items:center; gap:8px; font-size:13px; font-weight:600; color:#059669; cursor:pointer;">
        <input type="checkbox" id="nb01_exact" onchange="filterNB01()"> Csak garantált pontos koordinátás (N={n_pontos})
      </label>
    </div>

    <!-- Városrészek szűrő -->
    <div style="background:#f8fafc; padding:14px; border-radius:8px; border:1px solid #e2e8f0;">
      <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:8px;">
        <div style="font-weight:700; color:#334155; font-size:13px; text-transform:uppercase;">3. Városrészek Kijelölése:</div>
        <div style="font-size:11px;">
          <a href="javascript:void(0)" onclick="toggleAllVR(true)" style="color:#2563eb; margin-right:8px; text-decoration:none; font-weight:600;">Mind</a>
          <a href="javascript:void(0)" onclick="toggleAllVR(false)" style="color:#dc2626; text-decoration:none; font-weight:600;">Töröl</a>
        </div>
      </div>
      <div id="nb01_vr_chips" style="display:flex; flex-wrap:wrap; gap:6px; max-height:110px; overflow-y:auto; padding:4px;">
        {"".join([f'<label style="font-size:11px; background:#ffffff; border:1px solid #cbd5e1; border-radius:14px; padding:3px 8px; cursor:pointer;"><input type="checkbox" class="nb01_vr_cb" value="{vr}" checked onchange="filterNB01()"> {vr}</label>' for vr in vr_list])}
      </div>
    </div>
  </div>

  <!-- KPI Rács -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(180px, 1fr)); gap:12px; margin-bottom:16px;">
    <div style="background:#eff6ff; border:1px solid #93c5fd; border-radius:10px; padding:12px 16px; text-align:center;">
      <div style="font-size:11px; font-weight:700; color:#1e40af; text-transform:uppercase;">Szűrt Elemek</div>
      <div id="nb01_res_count" style="font-size:24px; font-weight:800; color:#1d4ed8; margin:3px 0;">-- db</div>
      <div style="font-size:11px; color:#2563eb;">Kiválasztott mintaelemszám</div>
    </div>

    <div style="background:#f0fdf4; border:1px solid #86efac; border-radius:10px; padding:12px 16px; text-align:center;">
      <div style="font-size:11px; font-weight:700; color:#166534; text-transform:uppercase;">Kínálati Ár (medián)</div>
      <div id="nb01_res_price" style="font-size:24px; font-weight:800; color:#15803d; margin:3px 0;">--</div>
      <div id="nb01_lbl_price_type" style="font-size:11px; color:#16a34a;">Vételár medián</div>
    </div>

    <div style="background:#faf5ff; border:1px solid #d8b4fe; border-radius:10px; padding:12px 16px; text-align:center;">
      <div style="font-size:11px; font-weight:700; color:#6b21a8; text-transform:uppercase;">Fajlagos Négyzetméterár</div>
      <div id="nb01_res_nm" style="font-size:24px; font-weight:800; color:#7e22ce; margin:3px 0;">-- Ft/m²</div>
      <div style="font-size:11px; color:#9333ea;">Szűrt minta mediánja (átlag alul)</div>
    </div>

    <div style="background:#fffbeb; border:1px solid #fde68a; border-radius:10px; padding:12px 16px; text-align:center;">
      <div style="font-size:11px; font-weight:700; color:#92400e; text-transform:uppercase;">Átlagos Alapterület</div>
      <div id="nb01_res_area" style="font-size:24px; font-weight:800; color:#b45309; margin:3px 0;">-- m²</div>
      <div style="font-size:11px; color:#d97706;">Fizikai lakásméret (medián)</div>
    </div>
  </div>

  <!-- Városrészi Összesítő Mini Táblázat -->
  <div style="overflow-x:auto;">
    <table id="nb01_stat_table" class="table table-bordered table-striped table-hover" style="width:100%; font-size:12px; margin-top:8px;">
      <thead>
        <tr style="background:#f1f5f9; color:#334155;">
          <th>Városrész</th>
          <th style="text-align:right;">Hirdetésszám</th>
          <th style="text-align:right;">Ár medián</th>
          <th style="text-align:right;">Ár/m² medián (átlag)</th>
          <th style="text-align:right;">Terület medián</th>
        </tr>
      </thead>
      <tbody></tbody>
    </table>
  </div>
</div>

<script>
(function() {{
  const data = {json.dumps(microdata)};
  const med = arr => {{ if (!arr.length) return 0; const s = [...arr].sort((a,b)=>a-b); const m = Math.floor(s.length/2); return s.length % 2 ? s[m] : (s[m-1]+s[m])/2; }};
  const mean = arr => arr.length ? arr.reduce((a,b)=>a+b,0)/arr.length : 0;

  window.toggleAllVR = function(state) {{
    document.querySelectorAll('.nb01_vr_cb').forEach(cb => cb.checked = state);
    filter();
  }};

  function filter() {{
    const type = document.querySelector('input[name="nb01_type"]:checked').value;
    const exactOnly = document.getElementById('nb01_exact').checked;
    const selectedVR = new Set();
    document.querySelectorAll('.nb01_vr_cb:checked').forEach(cb => selectedVR.add(cb.value));

    const rows = data.filter(r => (type === 'all' || r.t === type) && (!exactOnly || r.pt === 1) && selectedVR.has(r.vr));
    const P = rows.map(r => r.p), NM = rows.map(r => r.nm), A = rows.map(r => r.a);
    const totalN = rows.length;
    const vrMap = {{}};
    for (let r of rows) {{
      if (!vrMap[r.vr]) vrMap[r.vr] = {{ n:0, P:[], NM:[], A:[] }};
      vrMap[r.vr].n++; vrMap[r.vr].P.push(r.p); vrMap[r.vr].NM.push(r.nm); vrMap[r.vr].A.push(r.a);
    }}

    document.getElementById('nb01_res_count').innerText = totalN.toLocaleString('hu-HU') + ' db';
    if (type === 'kiado') {{
      document.getElementById('nb01_res_price').innerText = Math.round(med(P)/1000).toLocaleString('hu-HU') + ' ezer Ft/hó';
      document.getElementById('nb01_lbl_price_type').innerText = 'Havi bérleti díj medián (átlag: ' + Math.round(mean(P)/1000).toLocaleString('hu-HU') + ' ezer)';
    }} else {{
      document.getElementById('nb01_res_price').innerText = (med(P)/1e6).toFixed(1) + ' M Ft';
      document.getElementById('nb01_lbl_price_type').innerText = 'Kínálati vételár medián (átlag: ' + (mean(P)/1e6).toFixed(1) + ' M Ft)';
    }}
    document.getElementById('nb01_res_nm').innerText = Math.round(med(NM)).toLocaleString('hu-HU') + ' Ft/m² (átlag: ' + Math.round(mean(NM)).toLocaleString('hu-HU') + ')';
    document.getElementById('nb01_res_area').innerText = med(A).toFixed(1) + ' m²';

    const tbody = document.querySelector('#nb01_stat_table tbody');
    let rowsHtml = '';
    const vrSorted = Object.keys(vrMap).sort((a,b) => vrMap[b].n - vrMap[a].n);
    for (let vr of vrSorted) {{
      const v = vrMap[vr];
      const pMed = type === 'kiado' ? (Math.round(med(v.P)/1000).toLocaleString('hu-HU') + ' ezer') : (med(v.P)/1e6).toFixed(1) + ' M';
      rowsHtml += `<tr>
        <td><b>${{vr}}</b></td>
        <td style="text-align:right;">${{v.n}} db</td>
        <td style="text-align:right;">${{pMed}} <span style="color:#94a3b8; font-size:10px;">(átl. ${{type === 'kiado' ? Math.round(mean(v.P)/1000).toLocaleString('hu-HU') : (mean(v.P)/1e6).toFixed(1)}})</span></td>
        <td style="text-align:right;">${{Math.round(med(v.NM)).toLocaleString('hu-HU')}} Ft <span style="color:#94a3b8; font-size:10px;">(átl. ${{Math.round(mean(v.NM)).toLocaleString('hu-HU')}})</span></td>
        <td style="text-align:right;">${{med(v.A).toFixed(1)}} m²</td>
      </tr>`;
    }}
    tbody.innerHTML = rowsHtml || '<tr><td colspan="5" style="text-align:center; color:#94a3b8;">Nincs találat a kiválasztott szűrőkkel.</td></tr>';
  }}

  window.filterNB01 = filter;
  setTimeout(filter, 50);
}})();
</script>
'''
display(HTML(html_nb01))


## Összefoglalás

A fejezet zárásaként a legfontosabb, gyorsítótárból származó számokat egyetlen mondatfolyamba rendezzük — az alábbi cella minden értéket futásidőben illeszt be, így a szöveg a vizsgált területtől függetlenül mindig a tényleges mintát írja le.


In [ ]:
# Záró összefoglaló — minden szám a cache-ből, f-stringgel (nincs hardkódolt érték)
def _n(v):
    return f"{int(v):,}".replace(',', ' ')

m2 = D['minta'].set_index('mutato')['ertek']
lt3 = D['listing_type_ar'].set_index('listing_type')
vr2 = D['varosresz']
dec2 = D['decilisek']
top = vr2.iloc[0]; bot = vr2.iloc[-1]
c_leg = C['profil'].sort_values('N', ascending=False).iloc[0]

print(
    f"A vizsgált terület hirdetési adatbázisa {_n(m2['osszes'])} rekordot tartalmaz: "
    f"{_n(m2['elado'])} eladó és {_n(m2['kiado'])} kiadó hirdetést. "
    f"Ebből {_n(m2['pontos'])} rekord ({m2['pontos']/m2['osszes']*100:.1f}%) rendelkezik A-osztályú, épületszintű koordinátával; "
    f"az elemzési minta (pontos, eladó) {_n(m2['pontos_elado'])} lakás. "
    f"Az eladó fajlagos árak mediánja {huf_fmt(lt3.loc['elado', 'median'])}, átlaga {huf_fmt(lt3.loc['elado', 'atlag'])}; "
    f"a legalacsonyabb decilis {huf_fmt(dec2['nm_ar_huf'].iloc[0])}, a legmagasabb {huf_fmt(dec2['nm_ar_huf'].iloc[-1])} szinten zár. "
    f"A városrészek mediánár-rangsorát {top['varosresz']} vezeti ({huf_fmt(top['median'])}, N={int(top['N'])}), "
    f"és {bot['varosresz']} zárja ({huf_fmt(bot['median'])}, N={int(bot['N'])}). "
    f"A K-means tipológia {_n(n_clus)} megfigyelésen a legnagyobb klasztert {_n(c_leg['N'])} lakással "
    f"({c_leg['N']/n_clus*100:.0f}% részarány) azonosította."
)
